# Notebook 08 — Residual Blocks, Losses, and Objectives

This notebook explains the **weighted least-squares residual vector** that
the Levenberg-Marquardt optimiser minimises, and shows how scalar loss
components `f1..f4` are derived from it.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1  Residual vector structure

Levenberg-Marquardt minimises `||r(theta)||²`. The residual vector `r` has five blocks:

| Block | Content | Shape |
|---|---|---|
| 1 | `√(w_phospho · W_data) · (P_sim − P_data)` | `(N·T,)` |
| 2 | `√(w_abundance · W_prot) · (A_sim − A_data)` | `(K·T,)` |
| 3 | `√(reg_lambda) · theta` | `(dim,)` |
| 4 | `√(lambda_net) · L_alpha @ alpha` | `(M,)` |
| 5 | `√(w_mrna · W_mrna) · (R_sim − R_obs)` | `(n_rna,)` |

Block 5 is only included when mRNA data is available.


In [ ]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.data_loader import (
    load_site_data, load_rna_data, load_kinase_site_matrix,
    load_tf_network, build_tf_prot_weights, apply_scaling, row_normalize,
)

TIMEPOINTS = list(range(1, 15))   # 14 time points x1..x14

# ── phosphosite + protein abundance ─────────────────────────────────────────
sites, proteins, site_prot_idx, positions, t_phos, Y, A_data, A_proteins = \
    load_site_data(str(SAMPLE_DIR / "protephospho.csv"), TIMEPOINTS)

K = len(proteins)
N = len(sites)
T = len(t_phos)
print(f"proteins : {proteins}  (K={K})")
print(f"sites    : {sites}  (N={N})")
print(f"t_phos   : {t_phos}  (T={T})")
print(f"Y        : {Y.shape}   (N × T  phosphosite data)")
print(f"A_data   : {A_data.shape}  (K × T  protein abundance)")

# ── mRNA ─────────────────────────────────────────────────────────────────────
gene_ids, t_rna, rna_matrix = load_rna_data(
    str(SAMPLE_DIR / "mrna.csv"), timepoints=TIMEPOINTS
)
print(f"gene_ids : {gene_ids}  (n_genes={len(gene_ids)})")
print(f"rna_matrix: {rna_matrix.shape}  (n_genes × T)")

# ── kinase-site matrix ───────────────────────────────────────────────────────
K_site_kin, kinases = load_kinase_site_matrix(
    str(SAMPLE_DIR / "kinase_sites.tsv"), sites
)
M = len(kinases)
print(f"kinases  : {kinases}  (M={M})")
print(f"K_site_kin: {K_site_kin.shape}  (N × M)")

# ── kinase → protein index ───────────────────────────────────────────────────
kin_to_prot_idx = np.array([proteins.index(k) for k in kinases], dtype=int)
print(f"kin_to_prot_idx: {kin_to_prot_idx}")

# ── TF network ───────────────────────────────────────────────────────────────
tf_net = load_tf_network(str(SAMPLE_DIR / "tf_mrna.csv"), gene_ids=gene_ids)
tf_prot_weights = build_tf_prot_weights(tf_net, gene_ids, proteins)
print(f"tf_prot_weights: {tf_prot_weights.shape}  (K × n_genes)")

# ── scaled data ──────────────────────────────────────────────────────────────
P_scaled, _, _  = apply_scaling(Y)
A_scaled, _, _  = apply_scaling(A_data)
dims = ModelDims(K=K, M=M, N=N)

## 2  Scalar loss components

Each block contributes one **scalar** loss:

```
f1 = mean( W_data   · (P_sim − P_data)² )   # phosphosite fit
f2 = mean( W_prot   · (A_sim − A_data)² )   # protein abundance fit
f3 = reg_lambda · ‖theta‖²                  # L2 regularisation
     + lambda_net · ‖L_alpha @ alpha‖²       # kinase-network penalty
f4 = mean( W_mrna   · (R_sim − R_obs)² )    # mRNA fit (0 if absent)
```

**Total loss**: `J = w_phospho·f1 + w_abundance·f2 + w_reg·f3 + w_mrna·f4`

## 3  Weight matrices

`build_weight_matrices(t, Y, A_data)` produces per-site, per-timepoint weights
that downweight noisy or missing observations.

In [ ]:
from phoscrosstalk.weighting import build_weight_matrices

W_data, W_prot, W_mrna = build_weight_matrices(
    t=t_phos,
    Y=Y,
    A_data=A_data,
)
print("W_data shape:", W_data.shape, "  (N × T)")
print("W_prot shape:", W_prot.shape, "  (K × T)")
print("W_mrna shape:", W_mrna.shape, "  (unused here)")
print("W_data stats: min={:.3f}  max={:.3f}  mean={:.3f}".format(
    W_data.min(), W_data.max(), W_data.mean()))
print("W_prot stats: min={:.3f}  max={:.3f}  mean={:.3f}".format(
    W_prot.min(), W_prot.max(), W_prot.mean()))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
im0 = axes[0].imshow(W_data, aspect="auto", cmap="viridis")
axes[0].set_title("W_data  (N × T)"); axes[0].set_xlabel("Time"); axes[0].set_ylabel("Site")
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(sites, fontsize=7)
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(W_prot, aspect="auto", cmap="viridis")
axes[1].set_title("W_prot  (K × T)"); axes[1].set_xlabel("Time"); axes[1].set_ylabel("Protein")
axes[1].set_yticks(range(K)); axes[1].set_yticklabels(proteins)
plt.colorbar(im1, ax=axes[1])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_weight_heatmaps.png", dpi=100)
plt.show()
print("Saved 08_weight_heatmaps.png")


## 4  Simulate and compute residuals for a random theta

In [ ]:
from phoscrosstalk.optimization import create_bounds
from phoscrosstalk.derived_rates import make_k_act_fn, make_s_prod_fn
from phoscrosstalk.simulation import simulate

Cg = np.zeros((N, N)); Cl = np.zeros((N, N))
R_kin = row_normalize(K_site_kin.T); L_alpha = np.zeros((M, M))
receptor_mask_prot = np.zeros(K); receptor_mask_kin = np.zeros(M)

k_act_fn = make_k_act_fn(t_rna=t_rna, rna_data=rna_matrix,
                         tf_prot_weights=tf_prot_weights, K=K)
s_prod_fn = make_s_prod_fn(
    t_protein=t_phos, Y_data=P_scaled,
    R_kin_site=row_normalize(K_site_kin.T),
    kin_to_prot_idx=kin_to_prot_idx, K=K, M=M,
)

xl, xu, dim = create_bounds(K, M, N)
rng = np.random.default_rng(0)
theta_rand = rng.uniform(xl, xu)

P_sim, A_sim = simulate(
    t_arr=t_phos, P_data0=P_scaled, A_data0=A_scaled,
    theta=theta_rand, Cg=Cg, Cl=Cl,
    site_prot_idx=site_prot_idx, K_site_kin=K_site_kin, R=R_kin,
    L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
    receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
    mechanism="dist", k_act_fn=k_act_fn, s_prod_fn=s_prod_fn,
)
print("P_sim:", P_sim.shape, "  A_sim:", A_sim.shape)


In [ ]:
w_phospho   = 1.0
w_abundance = 1.0
reg_lambda  = 1e-3
lambda_net  = 1e-3

# Block 1: phosphosite residuals
r1 = (np.sqrt(w_phospho * W_data) * (P_sim - P_scaled)).ravel()

# Block 2: protein abundance residuals
r2 = (np.sqrt(w_abundance * W_prot) * (A_sim - A_scaled)).ravel()

# Block 3: L2 regularisation
r3 = np.sqrt(reg_lambda) * theta_rand

# Block 4: kinase-network regularisation (L_alpha is zero here)
K_start = 2*K + 2   # start of log_alpha block in theta
alpha_vals = theta_rand[K_start : K_start + M]
r4 = np.sqrt(lambda_net) * (L_alpha @ alpha_vals)

r_full = np.concatenate([r1, r2, r3, r4])
for name, r in [("Block 1 (phospho)",  r1),
                ("Block 2 (abundance)", r2),
                ("Block 3 (L2 reg)",   r3),
                ("Block 4 (net reg)",  r4),
                ("Total residual",     r_full)]:
    print(f"  {name:<25}  {r.shape[0]:5d} elements")


In [ ]:
f1 = float(np.mean(W_data * (P_sim - P_scaled)**2))
f2 = float(np.mean(W_prot * (A_sim - A_scaled)**2))
f3 = float(reg_lambda * np.sum(theta_rand**2))
f4 = 0.0
J  = w_phospho*f1 + w_abundance*f2 + f3 + f4
print("=" * 45)
print(f"  f1 (phosphosite fit):    {f1:.6f}")
print(f"  f2 (protein abund fit):  {f2:.6f}")
print(f"  f3 (regularisation):     {f3:.6f}")
print(f"  f4 (mRNA fit):           {f4:.6f}")
print(f"  J  (total loss):         {J:.6f}")
print("=" * 45)


## 5  Visualise the residual vector

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].bar(range(len(r_full)), r_full, width=1.0, color="steelblue", alpha=0.7)
axes[0].axhline(0, color="black", lw=0.5)
axes[0].set_xlabel("Residual index"); axes[0].set_ylabel("Value")
axes[0].set_title("Full residual vector  r(theta)")
for boundary, label in [
    (len(r1), "B2"), (len(r1)+len(r2), "B3"), (len(r1)+len(r2)+len(r3), "B4")
]:
    axes[0].axvline(boundary, color="red", lw=1, ls="--", alpha=0.7)
    axes[0].text(boundary + 2, axes[0].get_ylim()[1]*0.85, label,
                 fontsize=7, color="red")

axes[1].bar(["f1 phospho", "f2 protein", "f3 reg", "f4 mRNA"],
            [f1, f2, f3, f4],
            color=["tab:blue","tab:orange","tab:green","tab:red"])
axes[1].set_ylabel("Loss value"); axes[1].set_title("Scalar loss components")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_residual_vector.png", dpi=100)
plt.show()
print("Saved 08_residual_vector.png")


## 6  Loss transfer functions

| Type | Formula | Property |
|---|---|---|
| `mse` | `r²` | Standard MSE; sensitive to outliers |
| `pseudo_huber` | `δ²(√(1+(r/δ)²)−1)` | Quadratic near 0, linear for large r |
| `log_cosh` | `log(cosh(r))` | Smooth Huber approximation |

In [ ]:
r_vals = np.linspace(-5, 5, 400)
delta = 1.0
mse_loss     = r_vals**2
pseudo_huber = delta**2 * (np.sqrt(1 + (r_vals/delta)**2) - 1)
log_cosh_l   = np.log(np.cosh(r_vals))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(r_vals, mse_loss,     label="mse (r²)",          lw=2)
ax.plot(r_vals, pseudo_huber, label="pseudo_huber (δ=1)", lw=2, ls="--")
ax.plot(r_vals, log_cosh_l,   label="log_cosh",           lw=2, ls=":")
ax.set_xlabel("residual r"); ax.set_ylabel("loss(r)")
ax.set_ylim(-0.5, 10); ax.legend()
ax.set_title("Loss transfer functions")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_loss_functions.png", dpi=100)
plt.show()
print("Saved 08_loss_functions.png")


## 7  Why weighted least-squares?

In phosphoproteomics, sites differ widely in signal quality:

- **High-confidence site**: large `W_data[i,:]` → dominates `f1`
- **Noisy/low-intensity site**: small `W_data[i,:]` → contributes weakly
- **Outlier time point**: can be downweighted without discarding the site

**Regularisation balance:**
- `reg_lambda` small → flexible fit, risk of overfitting
- `reg_lambda` large → theta near 0, risk of underfitting
- Typical range: `1e-4` to `1e-1`

In [ ]:
for lam in [0.0001, 0.001, 0.01, 0.1]:
    f3_lam = lam * np.sum(theta_rand**2)
    print(f"  reg_lambda={lam:.4f}  f3={f3_lam:.6f}")
